In [14]:
!pip install psycopg2-binary

In [15]:
import requests
import pandas as pd
import psycopg2

API_URL = 'https://www.anapioficeandfire.com/api'

In [16]:
def get_all(endpoint, **filters):
    items = []
    page = 1
    while True:
        params = {'page': page, 'pageSize': 50, **filters}
        response = requests.get(f'{API_URL}/{endpoint}', params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        if not data:           # пустая страница значит данные закончились
            break
        items += data
        page += 1
    return items

In [17]:
books = get_all('books')
df_books = pd.DataFrame(books)
print(df_books.shape)
df_books.head()

(12, 11)


,url,name,isbn,authors,numberOfPages,publisher,country,mediaType,released,characters,povCharacters
0,https://www.anapioficeandfire.com/api/books/1,A Game of Thrones,978-0553103540,[George R. R. Martin],694,Bantam Books,United States,Hardcover,1996-08-01T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
1,https://www.anapioficeandfire.com/api/books/2,A Clash of Kings,978-0553108033,[George R. R. Martin],768,Bantam Books,United States,Hardback,1999-02-02T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
2,https://www.anapioficeandfire.com/api/books/3,A Storm of Swords,978-0553106633,[George R. R. Martin],992,Bantam Books,United States,Hardcover,2000-10-31T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
3,https://www.anapioficeandfire.com/api/books/4,The Hedge Knight,978-0976401100,[George R. R. Martin],164,Dabel Brothers Publishing,United States,GraphicNovel,2005-03-09T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...
4,https://www.anapioficeandfire.com/api/books/5,A Feast for Crows,978-0553801507,[George R. R. Martin],784,Bantam Books,United Status,Hardcover,2005-11-08T00:00:00,[https://www.anapioficeandfire.com/api/charact...,[https://www.anapioficeandfire.com/api/charact...


In [18]:
houses = get_all('houses')
df_houses = pd.DataFrame(houses)
print(df_houses.shape)
df_houses.head()

(444, 16)


,url,name,region,coatOfArms,words,titles,seats,currentLord,heir,overlord,founded,founder,diedOut,ancestralWeapons,cadetBranches,swornMembers
0,https://www.anapioficeandfire.com/api/houses/1,House Algood,The Westerlands,"A golden wreath, on a blue field with a gold b...",,[],[],,,https://www.anapioficeandfire.com/api/houses/229,,,,[],[],[]
1,https://www.anapioficeandfire.com/api/houses/2,House Allyrion of Godsgrace,Dorne,"Gyronny Gules and Sable, a hand couped Or",No Foe May Pass,[],[Godsgrace],https://www.anapioficeandfire.com/api/characte...,https://www.anapioficeandfire.com/api/characte...,https://www.anapioficeandfire.com/api/houses/285,,,,[],[],[https://www.anapioficeandfire.com/api/charact...
2,https://www.anapioficeandfire.com/api/houses/3,House Amber,The North,,,[],[],,,,,,,[],[],[]
3,https://www.anapioficeandfire.com/api/houses/4,House Ambrose,The Reach,"Or, semy of ants gules",Never Resting,[],[],https://www.anapioficeandfire.com/api/characte...,,https://www.anapioficeandfire.com/api/houses/398,,,,[],[],[https://www.anapioficeandfire.com/api/charact...
4,https://www.anapioficeandfire.com/api/houses/5,House Appleton of Appleton,The Reach,"Or, an apple tree eradicated proper fructed gu...",,[],[Appleton],,,https://www.anapioficeandfire.com/api/houses/398,,,,[],[],[]


### Дома, у которых есть девиз

Девиз (motto) в API Ice and Fire хранится в поле  words, поэтому для фильтрации использован параметр hasWords=true (параметр hasTitles из подсказки фильтрует дома по титулам, а не по девизам).

In [20]:
houses_words = get_all('houses', hasWords='true')
df_houses_words = pd.DataFrame(houses_words)
print(df_houses_words.shape)
df_houses_words[['name', 'region', 'words']].head()

(68, 16)


,name,region,words
0,House Allyrion of Godsgrace,Dorne,No Foe May Pass
1,House Ambrose,The Reach,Never Resting
2,House Arryn of the Eyrie,The Vale,As High as Honor
3,House Ashford of Ashford,The Reach,Our Sun Shines Bright
4,House Baratheon of Storm's End,The Stormlands,Ours is the Fury


In [21]:
print((df_houses_words['words'] == '').sum())   # должно быть 0

0


In [19]:
houses_titles = get_all('houses', hasTitles='true')
df_houses_titles = pd.DataFrame(houses_titles)
print(df_houses_titles.shape)

(165, 16)


In [22]:
conn = psycopg2.connect(
    host='hh-pgsql-public.ebi.ac.uk',
    port=5432,
    dbname='pfmegrnargs',
    user='reader',
    password='NWDMCE5xdipIjRrp',
)
cursor = conn.cursor()
print('Подключение установлено')

Подключение установлено


In [23]:
cursor.execute('SELECT * FROM rnc_database LIMIT 10')
rows = cursor.fetchall()
print(len(rows))
rows[:2]

10


[(5,
  datetime.datetime(2017, 5, 17, 0, 0),
  'RNACEN',
  'VEGA',
  98,
  'VEGA',
  'N',
  ' ',
  'VEGA',
  'PRJEB4568',
  None,
  None,
  None,
  0,
  0,
  'is a repository for high-quality gene models produced by the manual annotation of vertebrate genomes. Human and mouse data from Vega are merged into <a href="http://www.gencodegenes.org/" target="_blank">GENCODE</a>',
  'http://vega.sanger.ac.uk/',
  [{'upi': 'URS00000B15DA', 'taxid': 9606},
   {'upi': 'URS00000A54A6', 'taxid': 9606},
   {'upi': 'URS0000301B08', 'taxid': 9606}],
  [{'title': 'The GENCODE v7 catalog of human long noncoding RNAs: analysis of their gene structure, evolution, and expression.',
    'authors': 'Derrien T., Johnson R., Bussotti G., Tanzer A., Djebali S., Tilgner H., Guernec G., Martin D., Merkel A., Knowles DG. et al.',
    'journal': 'Genome Res. 22(9): 1775-1789 (2012 Sep)',
    'pubmed_id': '22955988'},
   {'title': 'GENCODE: the reference human genome annotation for The ENCODE Project',
    'authors

In [24]:
columns = [col[0] for col in cursor.description]
df_db = pd.DataFrame(rows, columns=columns)
df_db

,id,timestamp,userstamp,descr,current_release,full_descr,alive,for_release,display_name,project_id,avg_length,min_length,max_length,num_sequences,num_organisms,description,url,example,reference,last_import_date
0,5,2017-05-17 00:00:00.000000,RNACEN,VEGA,98,VEGA,N,,VEGA,PRJEB4568,NaN,NaN,NaN,0,0,is a repository for high-quality gene models p...,http://vega.sanger.ac.uk/,"[{'upi': 'URS00000B15DA', 'taxid': 9606}, {'up...",[{'title': 'The GENCODE v7 catalog of human lo...,NaT
1,26,2017-05-01 00:00:00.000000,RNACEN,GENCODE,450,GENCODE,N,,GENCODE,,889.0,32.0,205012.0,47677,2,produces high quality reference gene annotatio...,http://gencodegenes.org/,"[{'upi': 'URS00000B15DA', 'taxid': 9606}, {'up...",[{'title': 'GENCODE: the reference human genom...,NaT
2,25,2017-05-11 00:00:00.000000,RNACEN,ENSEMBL,1118,Ensembl,Y,,Ensembl,,1716.0,11.0,347561.0,3373550,267,is a genome browser for vertebrate genomes and...,http://ensembl.org/,"[{'upi': 'URS000025784F', 'taxid': 9606}, {'up...","[{'title': 'Ensembl 2017', 'authors': 'Aken BL...",2026-08-13 23:20:49.325656
3,1,2017-05-01 00:00:00.000000,RNACEN,ENA,1161,ENA,Y,,ENA,,412.0,10.0,900074.0,12086180,814855,provides a comprehensive record of the world's...,https://www.ebi.ac.uk/ena/browser/,"[{'upi': 'URS00002D0E0C', 'taxid': 10090}, {'u...",[{'title': 'The European Nucleotide Archive in...,NaT
4,59,2026-04-21 09:56:18.048915,RNACEN,CIRCATLAS,1090,circAtlas,Y,None,circAtlas,None,379.0,20.0,2808.0,2800521,10,: a gateway to 3 million standardized named ve...,https://ngdc.cncb.ac.cn/circatlas/tutorial1.php,None,[{'title': 'circAtlas 3.0: a gateway to 3 mill...,2026-05-03 04:33:18.130379
5,31,2017-05-01 00:00:00.000000,RNACEN,ENSEMBL_PLANTS,1119,Ensembl Plants,Y,,Ensembl Plants,,563.0,12.0,140059.0,167931,69,is a genome browser for plant genomes that com...,https://plants.ensembl.org/,"[{'upi': 'URS0000A77357', 'taxid': 3702}, {'up...",[{'title': 'Ensembl Genomes 2018: an integrate...,2026-08-13 23:23:46.972921
6,32,2017-05-01 00:00:00.000000,RNACEN,LNCBASE,248,LncBase,Y,,LncBase,,22.0,17.0,26.0,1352,2,experimentally verified and computationally pr...,http://www.microrna.gr/LncBase,"[{'upi': 'URS000075EAB0', 'taxid': 9606}, {'up...",[{'title': 'DIANA-LncBase v2: indexing microRN...,2018-10-04 15:46:52.689649
7,7,2017-05-01 00:00:00.000000,RNACEN,LNCRNADB,464,lncRNAdb,N,,lncRNAdb,PRJEB6238,3086.0,61.0,32753.0,62,10,is a database providing comprehensive annotati...,http://lncrnadb.org/,"[{'upi': 'URS00000478B7', 'taxid': 9606}, {'up...",[{'title': 'lncRNAdb: a reference database for...,2020-08-19 21:22:11.589150
8,57,2026-02-11 13:08:17.996559,RNACEN,CIRCPEDIA,1084,CIRCpedia,Y,None,CIRCpedia,None,975.0,11.0,340824.0,2530830,20,: an expanded repository to annotate both circ...,https://bits.fudan.edu.cn/circpediav3,None,[{'title': 'CIRCpedia v3: an interactive datab...,2026-04-13 22:41:17.759747
9,46,2017-05-01 00:00:00.000000,RNACEN,PIRBASE,1154,piRBase,Y,,piRBase,,27.0,15.0,40.0,480870,26,a database of various piRNA associated data to...,http://www.regulatoryrna.org/database/piRNA/,"[{'upi': 'URS0000061ED0', 'taxid': 9606}, {'up...",[{'title': 'piRBase: a comprehensive database ...,2026-09-18 12:20:09.699981


In [25]:
cursor.execute('''
    SELECT display_name, num_sequences, num_organisms, url
    FROM rnc_database
    LIMIT 10
''')
rows_cols = cursor.fetchall()

df_db_cols = pd.DataFrame(
    rows_cols,
    columns=['display_name', 'num_sequences', 'num_organisms', 'url']
)
df_db_cols

,display_name,num_sequences,num_organisms,url
0,VEGA,0,0,http://vega.sanger.ac.uk/
1,GENCODE,47677,2,http://gencodegenes.org/
2,Ensembl,3373550,267,http://ensembl.org/
3,ENA,12086180,814855,https://www.ebi.ac.uk/ena/browser/
4,circAtlas,2800521,10,https://ngdc.cncb.ac.cn/circatlas/tutorial1.php
5,Ensembl Plants,167931,69,https://plants.ensembl.org/
6,LncBase,1352,2,http://www.microrna.gr/LncBase
7,lncRNAdb,62,10,http://lncrnadb.org/
8,CIRCpedia,2530830,20,https://bits.fudan.edu.cn/circpediav3
9,piRBase,480870,26,http://www.regulatoryrna.org/database/piRNA/


In [26]:
cursor.close()
conn.close()